In [11]:
import pandas as pd
import arviz as az
from Functions import *

In [12]:
data=pd.read_excel(r"../Data_and_Preprocessing/pm25_BC_corrected.xlsx")
train=data.loc[data.label==0,:,]
device="cuda"

In [13]:
data

,Unnamed: 0.1,household,Unnamed: 0,flow,pm_final,sv_final,corrected_week,road,passive,Area,label,BC_Gaussion,BC_linear,BC_exp,BC_log,pm25_Gaussion,pm25_linear,pm25_exp,pm25_log
0,0,349,0,4.280,9.779,0.763,45.093,11475.472,1.073,129.469,1,0.917432,0.902661,0.889667,0.256274,11.758273,12.195733,11.978317,10.419316
1,1,146,1,3.925,23.367,1.813,44.980,20326.884,743.418,119.334,1,1.737960,1.740853,1.739927,3.854405,22.399847,22.189215,22.131170,22.988583
2,2,119,2,4.081,16.049,0.466,16.330,14945.838,721.708,150.379,1,0.489329,0.487835,0.487172,0.296438,16.852459,17.024964,17.019016,16.339483
3,3,379,3,4.032,16.246,0.748,21.960,24187.578,366.093,124.247,0,0.762269,0.761465,0.761246,0.610127,16.555918,16.622450,16.627049,16.360909
4,4,224,4,4.050,21.389,1.022,6.269,18357.834,0.000,94.688,1,1.052863,1.051039,1.050418,0.755310,22.034918,22.173639,22.178025,21.626328
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1115,1115,71,1115,4.030,6.364,0.736,43.327,22979.127,371.350,137.249,1,0.749139,0.748407,0.748212,0.607335,6.477610,6.502049,6.503836,6.406181
1116,1116,0,1116,3.942,44.469,1.280,47.346,14879.327,163.501,100.985,0,1.238535,1.240251,1.239918,2.167948,43.028439,42.715615,42.639077,43.910030
1117,1117,188,1117,3.782,14.995,1.303,45.377,15040.658,366.177,106.666,0,1.160862,1.162914,1.156143,-2.415400,13.359273,12.990744,12.804224,14.310297
1118,1118,362,1118,4.057,6.760,0.398,13.024,19799.986,747.378,134.070,1,0.411772,0.410943,0.410641,0.283775,6.993919,7.044163,7.045010,6.845642


In [14]:
train=make_periodic(train,"corrected_week",device) 


/home/michaelf/Seasonality_pyro_rewrite-Copy1/Seasonality_functions/Functions.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.loc[:,"theta"]=data.loc[:,variable]/data.loc[:,variable].max()*2*np.pi
/home/michaelf/Seasonality_pyro_rewrite-Copy1/Seasonality_functions/Functions.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.loc[:,"cos_theta"]=np.cos(data.loc[:,"theta"])
/home/michaelf/Seasonality_pyro_rewrite-Copy1/Seasonality_functions/Functions.py:18: SettingWithCopyWarning: 
A value is t

In [15]:
X_pm25,y_pm25=make_Xy(train,["cos_theta","sin_theta"],"pm25_Gaussion",device) 
nuts_kernel_pm25,gpr_pm25=model(X_pm25,y_pm25,device) 
pyro,gpr_pm25=train_model(X_pm25,nuts_kernel_pm25,gpr_pm25,device) 
torch.save(gpr_pm25,"../models/pm25_seasonality");

Warmup:   0%|                                         | 34/14000 [00:02,  7.68it/s, step size=2.62e-02, acc. prob=0.803]

KeyboardInterrupt: 

Warmup:   0%|                                         | 34/14000 [00:14,  7.68it/s, step size=2.62e-02, acc. prob=0.803]

In [ ]:
X_bc,y_bc=make_Xy(train,["cos_theta","sin_theta"],"BC_Gaussion",device) 
nuts_kernel_bc,gpr_bc=model(X_bc,y_bc,device,device) 
nuts_kernel_bc,gpr_bc,mcmc=train_model(X_bc,nuts_kernel_bc,gpr_bc,device) 
torch.save(gpr_bc,"../models/bc_seasonality");

In [6]:
gpr_pm25

GPRegression(
  (kernel): RBF()
)